# Session 4 Bioinformatics formats and validation



## Google Colab setup (optional)

Colab users should copy the repository folder to Google Drive, uncomment and
run the installation cell once per fresh runtime, and then uncomment the Drive
lines in the imports and paths cell below. Local Jupyter users can leave all
Colab-specific lines commented.


In [ ]:
# Google Colab only: uncomment and run once per fresh runtime.
# %pip install -q "biopython>=1.85,<2" "pandas>=2.2,<3" "pysam>=0.23,<0.24"


In [ ]:
from pathlib import Path
from urllib.parse import unquote

import pandas as pd
import pysam
from Bio import SeqIO

REPO_ROOT = None
# Google Colab only: uncomment these lines after copying the repository to Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# REPO_ROOT = Path("/content/drive/MyDrive/UIC_Bioinformatics_-_AI_for_Healthcare")

# Local Jupyter default: discover the repository from the current directory.
if "REPO_ROOT" not in globals():
    cwd = Path.cwd().resolve()
    REPO_ROOT = next((
        candidate for candidate in (cwd, *cwd.parents)
        if (candidate / "Module02_02" / "data").is_dir()
    ), None)

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Repository not found. In Colab, mount Drive and set REPO_ROOT above."
    )

module_dir = Path(REPO_ROOT) / "Module02_02"
data_dir = module_dir / "data" / "test"
haemophilus_dir = module_dir / "data" / "teaching" / "haemophilus_influenzae"
missing_dirs = [path for path in (data_dir, haemophilus_dir) if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError(f"Missing required data directories: {missing_dirs}")
print("Module directory:", module_dir.resolve())


## The file ecosystem

FASTA reference → FASTQ reads → SAM/BAM alignments → VCF variants.
GFF/GTF annotations and BED intervals add named biological features and
regions. Identifiers and coordinates connect the files.


In [ ]:
#Quick code to list the files of the directory.
for path in sorted(data_dir.iterdir()):
    if path.is_file():
        print(f"{path.name:30} {path.stat().st_size:>6} bytes")


## FASTA

Assemblers, reference databases, and sequence exports produce FASTA.
Headers begin with >; sequences can wrap. FASTA does not encode base
qualities, genomic coordinates, or sample genotypes.


In [ ]:
fasta_path = data_dir / "reference.fasta"
print("\n".join(fasta_path.read_text().splitlines()[:8]))


In [ ]:
fasta_records = list(SeqIO.parse(fasta_path, "fasta"))
#We use the SeqIO objet to parse the fasta files. We can acces different attributes (seq and id)
reference = {record.id: str(record.seq).upper() for record in fasta_records}
pd.DataFrame(
    {"record_id": key, "length": len(value), "alphabet": "".join(sorted(set(value)))}
    for key, value in reference.items()
)


In [ ]:
#Quick assertions to validate the sequences.
assert reference
assert len(reference) == len(set(reference))
for record_id, sequence in reference.items():
    assert sequence, f"Empty sequence: {record_id}"
    unexpected = set(sequence) - set("ACGTN")
    assert not unexpected, (record_id, unexpected)


## FASTQ

Sequencers and simulators produce reads plus encoded base-quality values.
Pair identifiers, compression, encoding, and sequence/quality agreement
matter. Four-line counting is a screen, not a universal parser.


In [ ]:
fastq_path = data_dir / "reads.fastq"
bad_fastq_path = data_dir / "reads_with_issue.fastq"
print("\n".join(fastq_path.read_text().splitlines()[:8]))


In [ ]:
def validate_fastq_four_line(path):
    errors = []
    lines = Path(path).read_text().splitlines()
    if len(lines) % 4:#% means modulus, so if the number of lines is not divisible by 4, then we have an error.
        errors.append({"line": len(lines), "rule": "record_boundary"})
    for index in range(0, len(lines) - 3, 4):
        header, sequence, separator, quality = lines[index:index + 4]#Keep it multiples of 4
        if not header.startswith("@"):
            errors.append({"line": index + 1, "rule": "header"})
        if not separator.startswith("+"):
            errors.append({"line": index + 3, "rule": "separator"})
        if len(sequence) != len(quality):
            errors.append({"line": index + 4, "rule": "quality_length"})
    return errors


**Which malformed record could pass these four rules?**

In [ ]:
print("valid fixture:", validate_fastq_four_line(fastq_path))
print("malformed fixture:", validate_fastq_four_line(bad_fastq_path))


In [ ]:
parsed_reads = list(SeqIO.parse(fastq_path, "fastq"))
assert parsed_reads
first_qualities = parsed_reads[0].letter_annotations["phred_quality"]#Get the quality scores for the first read
print(parsed_reads[0].id, len(parsed_reads[0]), min(first_qualities), max(first_qualities))


## SAM/BAM

Aligners place reads against a reference. SAM is generated as text; BAM is generated as compressed
binary (more often than not, you will work in BAM rather than SAM). Headers declare references and sorting.


Records has 11 fields, which include flags, reference, 1-based POS, MAPQ, CIGAR, mate fields, sequence, and quality.


In [ ]:
sam_path = data_dir / "alignments.sam"
sam_lines = sam_path.read_text().splitlines()
print("\n".join(sam_lines[:6]))


In [ ]:
def parse_sam_record(line):
    fields = line.rstrip().split("	")
    if len(fields) < 11:
        raise ValueError("SAM record has fewer than 11 mandatory fields")
    return {
        "qname": fields[0], "flag": int(fields[1]), "rname": fields[2],
        "pos": int(fields[3]), "mapq": int(fields[4]), "cigar": fields[5],
        "sequence": fields[9],
    }

sam_records = [parse_sam_record(line) for line in sam_lines if not line.startswith("@")]
pd.DataFrame(sam_records)


In [ ]:
reference_lengths = {name: len(sequence) for name, sequence in reference.items()}
sam_errors = []
for record in sam_records:
    if record["rname"] not in reference_lengths:
        sam_errors.append((record["qname"], "unknown_reference"))
    elif record["pos"] + len(record["sequence"]) - 1 > reference_lengths[record["rname"]]:
        sam_errors.append((record["qname"], "reference_bound"))
sam_errors


**Why does `grep` on SAM not test BAM readability?**

In [ ]:
test_bam_path = data_dir / "alignments.bam"
assert test_bam_path.is_file(), "Run the instructor fixture build before class"
with pysam.AlignmentFile(test_bam_path, "rb") as bam:#rb means read binary, which is the format of bam files.
    print("references:", bam.references)
    print("lengths:", bam.lengths)
    first_alignment = next(bam.fetch(until_eof=True))#Fetch the first alignment until the end of the file.
    print(first_alignment.query_name, first_alignment.flag, first_alignment.cigarstring)


CIGAR can make reference span differ from read length.

**Which CIGAR operations would higlight the underlying issue?**


## VCF

Variant callers compare sample evidence with a reference.

Metadata defines fields, #CHROM defines columns, REF must match the reference, ALT describes
the candidate change, and FILTER communicates caller decisions.

It has at least 8 fields.


In [ ]:
vcf_path = data_dir / "variants.vcf"
print("\n".join(vcf_path.read_text().splitlines()[:12]))


In [ ]:
def read_vcf_records(path):
    records = []
    for line_number, line in enumerate(Path(path).read_text().splitlines(), 1):
        if not line or line.startswith("#"):#Skip empty lines and commented lines (metadata within the VCF file)
            continue
        fields = line.split("	")
        if len(fields) < 8:
            raise ValueError(f"VCF line {line_number} has fewer than 8 fields")
        records.append({
            "line": line_number, "chrom": fields[0], "pos": int(fields[1]),
            "id": fields[2], "ref": fields[3], "alt": fields[4],
            "filter": fields[6],
        })
    return records

vcf_records = read_vcf_records(vcf_path)
pd.DataFrame(vcf_records)


**What information from INFO and sample columns is left out?**

In [ ]:
vcf_errors = []
for record in vcf_records:
    sequence = reference.get(record["chrom"])
    if sequence is None:
        vcf_errors.append((record["id"], "unknown_reference"))
        continue
    observed_ref = sequence[record["pos"] - 1:record["pos"] - 1 + len(record["ref"])]
    if observed_ref != record["ref"]:
        vcf_errors.append((record["id"], "reference_allele_mismatch"))
vcf_errors


In [ ]:
pd.DataFrame(vcf_errors, columns=["record_id", "rule"])


**What has a clean vcf_errors list actually established?**

## GFF3/GTF

Annotation pipelines describe features with nine fields.

GFF3 uses key=value attributes and Parent relationships.
GTF commonly uses quoted key-value pairs.

Coordinates are normally 1-based and closed. Strand and CDS phase affect sequence extraction. Always check which kind of annotation format you are using.



In [ ]:
gff_path = data_dir / "annotation.gff3"
print("\n".join(gff_path.read_text().splitlines()[:8]))


In [ ]:
def parse_gff3_attributes(text):
    attributes = {}
    for item in text.split(";"):#Split the attributes by semicolon as it is the standard delimiter for GFF3 attributes
        if not item:
            continue
        key, value = item.split("=", 1) if "=" in item else (item, "")
        attributes[key] = unquote(value)
    return attributes


In [ ]:
gff_rows = []
for line_number, line in enumerate(gff_path.read_text().splitlines(), 1):
    if not line or line.startswith("#"):
        continue
    fields = line.split("	")
    gff_rows.append({
        "line": line_number, "seqid": fields[0], "type": fields[2],
        "start": int(fields[3]), "end": int(fields[4]),
        "strand": fields[6], "phase": fields[7],
        "attributes": parse_gff3_attributes(fields[8]),
    })#Parse the GFF3 row using all the fields and the attributes parser function defined above.
pd.DataFrame([{**row, "attributes": str(row["attributes"])} for row in gff_rows])


**Why split only on the first equals sign?**


In [ ]:
gff_ids = {row["attributes"]["ID"] for row in gff_rows if "ID" in row["attributes"]}
gff_errors = []
for row in gff_rows:
    if row["start"] < 1 or row["end"] < row["start"]:
        gff_errors.append((row["line"], "invalid_interval"))
    for parent in row["attributes"].get("Parent", "").split(","):
        if parent and parent not in gff_ids:
            gff_errors.append((row["line"], "missing_parent"))
gff_errors


**What is the length of feature 101–101?**

## Using a real GFF3/protein-FASTA case
Count CDS rows, distinct `protein_id` values, and protein FASTA records. A row is
not automatically one feature or one protein product.


In [ ]:
hi_gff_path = haemophilus_dir / "annotation.gff3"
hi_protein_path = haemophilus_dir / "proteins.faa"
hi_cds_rows = []
for line_number, line in enumerate(hi_gff_path.read_text().splitlines(), 1):
    if not line or line.startswith("#"):
        continue
    fields = line.split("	")
    if len(fields) == 9 and fields[2] == "CDS":
        attributes = parse_gff3_attributes(fields[8])
        hi_cds_rows.append({
            "line": line_number,
            "seqid": fields[0],
            "start": int(fields[3]),
            "end": int(fields[4]),
            "strand": fields[6],
            "phase": fields[7],
            "protein_id": attributes.get("protein_id"),
            "translation_table": attributes.get("transl_table"),
        })

hi_proteins = {record.id: record for record in SeqIO.parse(hi_protein_path, "fasta")}
hi_protein_ids = {row["protein_id"] for row in hi_cds_rows if row["protein_id"]}
summary = {
    "CDS_rows": len(hi_cds_rows),
    "distinct_GFF_protein_ids": len(hi_protein_ids),
    "protein_FASTA_records": len(hi_proteins),
    "GFF_ids_missing_from_FASTA": len(hi_protein_ids - set(hi_proteins)),
}
assert summary["CDS_rows"] == 1617
assert summary["distinct_GFF_protein_ids"] == 1610
assert summary["protein_FASTA_records"] == 1610
summary


**Why does a row count differ from the protein count?**

In [ ]:
hi_cds_table = pd.DataFrame(hi_cds_rows)
segment_counts = hi_cds_table.groupby("protein_id").size().sort_values(ascending=False)
display(hi_cds_table.groupby("strand").size().rename("CDS_rows"))
display(segment_counts.head().rename("CDS_rows_per_protein"))

multisegment_id = segment_counts.index[0]
display(hi_cds_table.loc[hi_cds_table["protein_id"] == multisegment_id])
print("Annotated protein length:", len(hi_proteins[multisegment_id]))


**How can you calculate the annotated protein length alternatively?**

## GTF comparison

GTF also has nine tab-separated fields, but the attribute grammar differs.
Do not reuse a GFF3 key=value parser without testing it against GTF text.


In [ ]:
example_gtf = 'gene_id "gene_0001"; transcript_id "tx_0001";'
print(example_gtf)
print("A GFF3 split on = would not recover these quoted values correctly.")


## BED theory

BED represents intervals such as targets, peaks, and windows. Start is
0-based and included; end is excluded. Therefore end must be greater than
start, and a one-base GFF3 feature 101–101 becomes BED 100–101.


In [ ]:
bed_path = data_dir / "targets.bed"
print(bed_path.read_text())


In [ ]:
bed_rows = []
bed_errors = []
for line_number, line in enumerate(bed_path.read_text().splitlines(), 1):
    if not line or line.startswith(("#", "track", "browser")):
        continue
    fields = line.split("	")
    chrom, start, end = fields[0], int(fields[1]), int(fields[2])
    bed_rows.append((chrom, start, end, fields[3] if len(fields) > 3 else "."))
    if start < 0 or end <= start:
        bed_errors.append((line_number, "invalid_interval"))
print("rows:", bed_rows)
print("errors:", bed_errors)


In [ ]:
def gff_to_bed(start_1based, end_1based):
    if start_1based < 1 or end_1based < start_1based:#Condition check to convert gff 1-based interval to 0-based interval for bed
        raise ValueError("Invalid 1-based closed interval")
    return start_1based - 1, end_1based

assert gff_to_bed(101, 101) == (100, 101)


In [ ]:
def overlaps(start_a, end_a, start_b, end_b):
    """Check if two intervals overlap at some point."""
    return start_a < end_b and start_b < end_a

assert overlaps(10, 20, 19, 25)
assert not overlaps(10, 20, 20, 25)


Our currect function reject negative starts and intervals with end <= start. Half-open intervals touching at an endpoint do not overlap.

**Do [10,20) and [20,25) overlap? Which assertion proves it?**